In [25]:
"""
=============================================================================
 HYBRID QUANTUM ML – CALIFORNIA WILDFIRE RISK PREDICTION  (v7)
 Architecture: Local Train + SV1 Prediction

 Strategy:
   PHASE 1 — TRAIN on local PennyLane default.qubit (free, fast)
     - Identical circuit, identical Adam optimizer
     - 4-qubit statevector sim is numerically identical to SV1
     - No AWS costs during training
     - Saves best weights to disk

   PHASE 2 — PREDICT on Amazon Braket SV1 (minimal cost ~$1.50)
     - Loads saved weights
     - Runs test evaluation on SV1
     - Runs 2023 risk predictions on SV1
     - Confirms weights transfer cleanly to real AWS backend

 Cost vs full Hybrid Jobs:  ~$1.50  vs  ~$85
 Accuracy difference:        none   (same circuit, same statevector math)
=============================================================================
"""

import os, time, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import boto3
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, roc_auc_score,
                             confusion_matrix, accuracy_score, f1_score)
import pennylane as qml
from pennylane import numpy as pnp
import json

In [26]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
DATASET_PATH   = "/home/ec2-user/SageMaker/Code/task1a/Dataset2.csv"
WEIGHTS_PATH   = "results/best_weights.npz"      # saved after training
S3_BUCKET_NAME = "amazon-braket-qmlsv1"                            # None = auto-discover

# ── Hyperparameters ───────────────────────────────────────────────────────
N_QUBITS     = 4
N_LAYERS     = 3
N_EPOCHS     = 150     # more epochs now that gradient actually moves
BATCH_SIZE   = 32
LR           = 0.05
N_SEEDS      = 3
INIT_SCALE   = np.pi   # FIX: was 0.05 — caused barren plateau
L2_BIAS      = 0.001   # reduced — was over-constraining bias
TOTAL_PARAMS = N_LAYERS * N_QUBITS * 3   # 36

print("=" * 70)
print("  HYBRID QML v7 — LOCAL TRAIN + SV1 PREDICT")
print("=" * 70)

os.makedirs("results", exist_ok=True)


  HYBRID QML v7 — LOCAL TRAIN + SV1 PREDICT


In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# AWS SETUP (only needed for Phase 2 — set up early to fail fast if broken)
# ─────────────────────────────────────────────────────────────────────────────
print("\n[0/10] Checking AWS credentials for Phase 2…")
try:
    boto3.setup_default_session(region_name="us-east-1")
    sts      = boto3.client("sts")
    identity = sts.get_caller_identity()
    print(f"   ✅ AWS Account : {identity['Account']}")
    print(f"   ✅ Role        : {identity['Arn'].split('/')[-1]}")
    AWS_OK = True
except Exception as e:
    print(f"   ⚠  AWS not available ({e}) — Phase 2 will be skipped")
    AWS_OK = False

if AWS_OK:
    if S3_BUCKET_NAME is None:
        s3      = boto3.client("s3")
        buckets = [b["Name"] for b in s3.list_buckets().get("Buckets", [])]
        braket_buckets = [b for b in buckets if "braket" in b.lower()]
        S3_BUCKET_NAME = braket_buckets[0] if braket_buckets else (buckets[0] if buckets else None)
    print(f"   ✅ S3 bucket   : {S3_BUCKET_NAME}")



[0/10] Checking AWS credentials for Phase 2…
   ✅ AWS Account : 517306214684
   ✅ Role        : Participant
   ✅ S3 bucket   : amazon-braket-qmlsv1


In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. LOAD
# ─────────────────────────────────────────────────────────────────────────────
print("\n[1/10] Loading Dataset2.csv…")
raw = pd.read_csv(DATASET_PATH, low_memory=False)
print(f"   Raw shape: {raw.shape}")


[1/10] Loading Dataset2.csv…
   Raw shape: (125476, 30)


In [9]:

# ─────────────────────────────────────────────────────────────────────────────
# 2. REAL NEGATIVES
# ─────────────────────────────────────────────────────────────────────────────
print("\n[2/10] Building annual climate baseline…")
CLIMATE_COLS = ["avg_tmax_c", "avg_tmin_c", "tot_prcp_mm"]

climate_rows = raw[raw["OBJECTIVE"].isna()].copy()
climate_rows["parsed_year"] = climate_rows["year_month"].str[:4].astype(float)
climate_rows["zip"] = climate_rows["zip"].astype(float)

annual_climate = (
    climate_rows[climate_rows["parsed_year"].between(2018, 2021)]
    .groupby(["zip", "parsed_year"])[CLIMATE_COLS]
    .mean().reset_index()
    .rename(columns={"parsed_year": "year"})
)
annual_climate["wildfire"] = 0
print(f"   Climate rows: {len(annual_climate)} | ZIPs: {annual_climate['zip'].nunique()}")



[2/10] Building annual climate baseline…
   Climate rows: 10372 | ZIPs: 2593


In [10]:

# ─────────────────────────────────────────────────────────────────────────────
# 3. POSITIVES
# ─────────────────────────────────────────────────────────────────────────────
print("\n[3/10] Building fire positive records…")
fires = raw[raw["OBJECTIVE"] == 1].copy()
fires["parsed_year"] = pd.to_datetime(fires["Alarm_Date2"], errors="coerce").dt.year
fires = fires.dropna(subset=["zip", "parsed_year"])
fires["zip"] = fires["zip"].astype(float)

fire_agg = (
    fires[fires["parsed_year"].between(2018, 2021)]
    .groupby(["zip", "parsed_year"])[CLIMATE_COLS]
    .mean().reset_index()
    .rename(columns={"parsed_year": "year"})
)
fire_agg["wildfire"] = 1
print(f"   Fire rows: {len(fire_agg)} | ZIPs: {fire_agg['zip'].nunique()}")



[3/10] Building fire positive records…
   Fire rows: 879 | ZIPs: 496


In [11]:

# ─────────────────────────────────────────────────────────────────────────────
# 4. MERGE
# ─────────────────────────────────────────────────────────────────────────────
print("\n[4/10] Merging…")
fire_keys   = set(zip(fire_agg["zip"], fire_agg["year"]))
fire_lookup = fire_agg.set_index(["zip", "year"])[CLIMATE_COLS]

df = annual_climate.copy()
df["wildfire"] = df.apply(
    lambda r: 1 if (r["zip"], r["year"]) in fire_keys else 0, axis=1
)
for col in CLIMATE_COLS:
    mask = df.apply(lambda r: (r["zip"], r["year"]) in fire_keys, axis=1)
    df.loc[mask, col] = df.loc[mask].apply(
        lambda r: fire_lookup.loc[(r["zip"], r["year"]), col]
        if (r["zip"], r["year"]) in fire_lookup.index else r[col], axis=1
    )

df = df.dropna(subset=CLIMATE_COLS)
df[CLIMATE_COLS] = df[CLIMATE_COLS].fillna(df[CLIMATE_COLS].median())
print(f"   Dataset: {df.shape} | fire={df['wildfire'].sum()} | "
      f"no-fire={(df['wildfire']==0).sum()} | "
      f"balance={df['wildfire'].mean()*100:.1f}%")


[4/10] Merging…
   Dataset: (10372, 6) | fire=871 | no-fire=9501 | balance=8.4%


In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# 5. 2023 PREDICTION SET
# ─────────────────────────────────────────────────────────────────────────────
df_2023 = annual_climate[annual_climate["year"] == 2021].copy()
df_2023["year"] = 2023
df_2023["avg_tmax_c"]  *= 1.02
df_2023["avg_tmin_c"]  *= 1.01
df_2023["tot_prcp_mm"] *= 0.97


In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# 6. SPLIT & SCALE
# ─────────────────────────────────────────────────────────────────────────────
print("\n[5/10] Splitting and scaling…")
X_all, y_all = df[CLIMATE_COLS].values, df["wildfire"].values

X_tmp,   X_test, y_tmp,   y_test = train_test_split(
    X_all, y_all, test_size=0.20, random_state=42, stratify=y_all)
X_train, X_val,  y_train, y_val  = train_test_split(
    X_tmp, y_tmp, test_size=0.25, random_state=42, stratify=y_tmp)

scaler    = MinMaxScaler(feature_range=(0, np.pi))
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)
X_pred_s  = scaler.transform(df_2023[CLIMATE_COLS].values)

def pad4(X): return np.hstack([X, X[:, :1]])
X_train_q, X_val_q = pad4(X_train_s), pad4(X_val_s)
X_test_q,  X_pred_q = pad4(X_test_s),  pad4(X_pred_s)

print(f"   Train:{X_train_q.shape} Val:{X_val_q.shape} "
      f"Test:{X_test_q.shape} Pred:{X_pred_q.shape}")

# ─────────────────────────────────────────────────────────────────────────────
# PHASE 1 — LOCAL TRAINING  (default.qubit, free)
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "█" * 70)
print("  PHASE 1 — LOCAL TRAINING  (default.qubit, free)")
print("█" * 70)

dev_local = qml.device("default.qubit", wires=N_QUBITS)

# ── Circuit: qubit-0 only — used for training (unchanged) ─────────────────
@qml.qnode(dev_local)
def quantum_circuit_local(inputs, weights):
    for l in range(N_LAYERS):
        for i in range(N_QUBITS):
            qml.RX(inputs[i], wires=i)
            qml.RZ(inputs[i], wires=i)
        for i in range(N_QUBITS):
            qml.Rot(weights[l, i, 0], weights[l, i, 1], weights[l, i, 2], wires=i)
        for i in range(N_QUBITS - 1):
            qml.CNOT(wires=[i, i + 1])
        qml.CNOT(wires=[N_QUBITS - 1, 0])
    return qml.expval(qml.PauliZ(0))

# ── FIX: Multi-qubit circuit — all 4 EVs used for downstream classifiers ──
@qml.qnode(dev_local)
def quantum_circuit_local_multi(inputs, weights):
    """Same circuit, but returns EV from all 4 qubits for richer features."""
    for l in range(N_LAYERS):
        for i in range(N_QUBITS):
            qml.RX(inputs[i], wires=i)
            qml.RZ(inputs[i], wires=i)
        for i in range(N_QUBITS):
            qml.Rot(weights[l, i, 0], weights[l, i, 1], weights[l, i, 2], wires=i)
        for i in range(N_QUBITS - 1):
            qml.CNOT(wires=[i, i + 1])
        qml.CNOT(wires=[N_QUBITS - 1, 0])
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -50, 50)))

def predict_proba_local(X_q, weights, scale, bias):
    probs = []
    for x in X_q:
        ev = float(quantum_circuit_local(x, weights))
        probs.append(sigmoid(float(scale) * ev + float(bias)))
    return np.array(probs)

def bce_weighted_local(weights, scale, bias, X_q, y, pos_w):
    p   = np.clip(predict_proba_local(X_q, weights, scale, bias), 1e-7, 1 - 1e-7)
    w   = np.where(y == 1, pos_w, 1.0)
    bce = -np.mean(w * (y * np.log(p) + (1 - y) * np.log(1 - p)))
    l2  = L2_BIAS * float(bias) ** 2
    return bce + l2

def cost_fn_local(params, X_q, y, pos_w):
    weights, scale, bias = params
    return bce_weighted_local(weights, scale, bias, X_q, y, pos_w)

def balanced_batch(X_q, y, bs):
    pos = np.where(y == 1)[0]; neg = np.where(y == 0)[0]; h = bs // 2
    idx = np.concatenate([
        np.random.choice(pos, min(h, len(pos)), replace=True),
        np.random.choice(neg, min(h, len(neg)), replace=True)
    ])
    np.random.shuffle(idx)
    return X_q[idx], y[idx]

# ── Train ──────────────────────────────────────────────────────────────────
print(f"\n[6/10] Training locally ({N_SEEDS} seeds × {N_EPOCHS} epochs)…")
pos_w = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
print(f"   Positive class weight: {pos_w:.2f}x")

best_val_loss      = np.inf
best_weights       = None
best_scale_val     = 1.0
best_bias_val      = 0.0
all_loss_histories = []

for seed in range(N_SEEDS):
    np.random.seed(seed * 13 + 7)

    # FIX: initialize in [-pi, pi] — breaks barren plateau
    weights = pnp.array(
        np.random.uniform(-INIT_SCALE, INIT_SCALE, (N_LAYERS, N_QUBITS, 3)),
        requires_grad=True
    )
    scale = pnp.array(1.0, requires_grad=True)
    # FIX: wider bias initialization
    bias  = pnp.array(np.random.uniform(-0.5, 0.5), requires_grad=True)
    opt   = qml.AdamOptimizer(stepsize=LR)

    loss_hist = []
    t0 = time.time()
    print(f"\n   Seed {seed+1}/{N_SEEDS}")

    for epoch in range(N_EPOCHS):
        Xb, yb = balanced_batch(X_train_q, y_train, BATCH_SIZE)
        params = (weights, scale, bias)
        params, loss = opt.step_and_cost(
            lambda p: cost_fn_local(p, Xb, yb, pos_w), params
        )
        weights, scale, bias = params
        # FIX: allow scale to go up to 10 — more room to amplify signal
        scale = pnp.array(float(np.clip(float(scale), 0.1, 10.0)), requires_grad=True)
        loss_hist.append(float(loss))

        if (epoch + 1) % 25 == 0:
            vl = bce_weighted_local(weights, scale, bias, X_val_q, y_val, pos_w)
            print(f"     Epoch {epoch+1:3d}/{N_EPOCHS} | "
                  f"Train: {float(loss):.4f} | Val: {float(vl):.4f} | "
                  f"scale={float(scale):.3f} bias={float(bias):.3f} | "
                  f"Time: {time.time()-t0:.0f}s")

    vl_final = bce_weighted_local(weights, scale, bias, X_val_q, y_val, pos_w)
    all_loss_histories.append(loss_hist)
    print(f"   Seed {seed+1} final val loss: {float(vl_final):.4f}")

    if float(vl_final) < best_val_loss:
        best_val_loss  = float(vl_final)
        best_weights   = weights.copy()
        best_scale_val = float(scale)
        best_bias_val  = float(bias)
        print(f"   ✅ New best (val loss: {best_val_loss:.4f})")

print(f"\n   Best val loss: {best_val_loss:.4f} | "
      f"scale={best_scale_val:.3f} bias={best_bias_val:.3f}")

np.savez(
    WEIGHTS_PATH,
    weights=np.array(best_weights),
    scale=np.array([best_scale_val]),
    bias=np.array([best_bias_val]),
    loss_histories=np.array(all_loss_histories),
)
print(f"\n   Weights saved -> {WEIGHTS_PATH}")



[5/10] Splitting and scaling…
   Train:(6222, 4) Val:(2075, 4) Test:(2075, 4) Pred:(2593, 4)

██████████████████████████████████████████████████████████████████████
  PHASE 1 — LOCAL TRAINING  (default.qubit, free)
██████████████████████████████████████████████████████████████████████

[6/10] Training locally (3 seeds × 150 epochs)…
   Positive class weight: 10.90x

   Seed 1/3
     Epoch  25/150 | Train: 3.3678 | Val: 1.3842 | scale=1.000 bias=0.269 | Time: 52s
     Epoch  50/150 | Train: 3.3632 | Val: 1.3842 | scale=1.000 bias=0.269 | Time: 100s
     Epoch  75/150 | Train: 3.5652 | Val: 1.3842 | scale=1.000 bias=0.269 | Time: 148s
     Epoch 100/150 | Train: 3.4800 | Val: 1.3842 | scale=1.000 bias=0.269 | Time: 197s
     Epoch 125/150 | Train: 3.5688 | Val: 1.3842 | scale=1.000 bias=0.269 | Time: 257s
     Epoch 150/150 | Train: 3.3504 | Val: 1.3842 | scale=1.000 bias=0.269 | Time: 323s
   Seed 1 final val loss: 1.3842
   ✅ New best (val loss: 1.3842)

   Seed 2/3
     Epoch  25/150

In [14]:
# ── Threshold tuning (locally, free) ─────────────────────────────────────
print("\n[7/10] Tuning threshold on validation set (local)…")
val_probs = predict_proba_local(X_val_q, best_weights, best_scale_val, best_bias_val)
print(f"   Val prob range: [{val_probs.min():.3f}, {val_probs.max():.3f}]  "
      f"mean={val_probs.mean():.3f}")

best_t, best_f1 = 0.5, 0.0
for t in np.arange(0.20, 0.81, 0.05):
    preds = (val_probs >= t).astype(int)
    if preds.sum() == 0: continue
    f = f1_score(y_val, preds, zero_division=0)
    if f > best_f1:
        best_f1, best_t = f, t
THRESHOLD = best_t
print(f"   Best threshold: {THRESHOLD:.2f} (val F1={best_f1:.4f})")

print("\n  PHASE 1 COMPLETE — weights saved, threshold tuned, $0 spent on AWS")




[7/10] Tuning threshold on validation set (local)…
   Val prob range: [0.355, 0.553]  mean=0.440
   Best threshold: 0.50 (val F1=0.2939)

  PHASE 1 COMPLETE — weights saved, threshold tuned, $0 spent on AWS


In [6]:
'''import sagemaker

session = sagemaker.Session()
print(session.default_bucket())'''

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:4                                                                                    │
│                                                                                                  │
│   1 import sagemaker                                                                             │
│   2                                                                                              │
│   3 session = sagemaker.Session()                                                                │
│ ❱ 4 print(session.default_bucket())                                                              │
│   5                                                                                              │
│                                                                                                  │
│ /home/ec2-user/SageMaker/qml_2025/lib/python3.10/site-packages/sagemaker/session.py:599 in       │
│ default_bucket                                                                                   │
│                                                                                                  │
│    596 │   │                                                                                     │
│    597 │   │   default_bucket = self._default_bucket_name_override                               │
│    598 │   │   if not default_bucket:                                                            │
│ ❱  599 │   │   │   default_bucket = generate_default_sagemaker_bucket_name(self.boto_session)    │
│    600 │   │   │   self._default_bucket_set_by_sdk = True                                        │
│    601 │   │                                                                                     │
│    602 │   │   self._create_s3_bucket_if_it_does_not_exist(                                      │
│                                                                                                  │
│ /home/ec2-user/SageMaker/qml_2025/lib/python3.10/site-packages/sagemaker/session.py:8329 in      │
│ generate_default_sagemaker_bucket_name                                                           │
│                                                                                                  │
│   8326 │   region = boto_session.region_name                                                     │
│   8327 │   account = boto_session.client(                                                        │
│   8328 │   │   "sts", region_name=region, endpoint_url=sts_regional_endpoint(region)             │
│ ❱ 8329 │   ).get_caller_identity()["Account"]                                                    │
│   8330 │   return "sagemaker-{}-{}".format(region, account)                                      │
│   8331                                                                                           │
│   8332                                                                                           │
│                                                                                                  │
│ /home/ec2-user/SageMaker/qml_2025/lib/python3.10/site-packages/botocore/client.py:602 in         │
│ _api_call                                                                                        │
│                                                                                                  │
│    599 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    600 │   │   │   │   )                                                                         │
│    601 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  602 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    603 │   │                                                                                     │
│    604 │   │   _api_call.__name__ = str(py_operation_name) 

In [27]:
print("y_test shape:", y_test.shape)
print("X_test_q shape:", X_test_q.shape)
print(len(y_test), len(X_test_q))  # these must match
print("test_probs shape:", test_probs.shape)

y_test shape: (2075,)
X_test_q shape: (2075, 4)
2075 2075
test_probs shape: (4,)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#
#  PHASE 2 — SV1 PREDICTIONS  (AWS Braket, ~$1.50)
# 
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "█" * 70)
print("  PHASE 2 — SV1 PREDICTIONS  (Amazon Braket, ~$1.50)")
print("█" * 70)

if not AWS_OK:
    print("\n  ⚠  AWS credentials not available — skipping Phase 2")
    print("     Re-run with valid credentials to execute SV1 predictions")
else:
    # ── Load weights saved in Phase 1 ─────────────────────────────────────
    print(f"\n[8/10] Loading weights from {WEIGHTS_PATH}…")
    ckpt           = np.load(WEIGHTS_PATH)
    sv1_weights    = pnp.array(ckpt["weights"],       requires_grad=False)
    sv1_scale      = float(ckpt["scale"][0])
    sv1_bias       = float(ckpt["bias"][0])
    print(f"   Weights loaded | scale={sv1_scale:.3f} bias={sv1_bias:.3f}")

    # ── SV1 device — vectorized (batch) circuit ────────────────────────────
    print("\n[9/10] Building SV1 device…")

    from braket.tracking import Tracker   # cost tracker

    DEVICE_ARN = "arn:aws:braket:::device/quantum-simulator/amazon/sv1"
    S3_FOLDER  = (S3_BUCKET_NAME, "wildfire-sv1-results/")

    dev_sv1 = qml.device(
        "braket.aws.qubit",
        device_arn=DEVICE_ARN,
        wires=N_QUBITS,
        s3_destination_folder=S3_FOLDER,
        parallel=True,
        max_parallel=30,
        shots=None,       # exact statevector
    )

    # Vectorized circuit — inputs shape (Batch, 4)
    @qml.qnode(dev_sv1)
    def quantum_circuit_sv1(inputs, weights):
        for l in range(N_LAYERS):
            for i in range(N_QUBITS):
                qml.RX(inputs[i], wires=i)
                qml.RZ(inputs[i], wires=i)
            for i in range(N_QUBITS):
                qml.Rot(weights[l, i, 0], weights[l, i, 1], weights[l, i, 2], wires=i)
            for i in range(N_QUBITS - 1):
                qml.CNOT(wires=[i, i + 1])
            qml.CNOT(wires=[N_QUBITS - 1, 0])
        return qml.expval(qml.PauliZ(0))

    def predict_proba_sv1(X_q, weights, scale, bias):
        """SV1 prediction — single batched call, returns (Batch,) array."""
        X_q = np.array(X_q)                          # force numpy, shape (2075, 4)
        evs = []
        total = X_q.shape[0]
        print(f"   Running {X_q.shape[0]} circuits...")
        for i in range(X_q.shape[0]):                 # iterate over 2075 rows
            x = X_q[i, :]                             # shape (4,)
            ev = quantum_circuit_sv1(pnp.array(x), weights)
            evs.append(float(ev))
            if (i + 1) % 50 == 0:                        # print every 50 circuits
                print(f"   Progress: {i+1}/{total} ({(i+1)/total*100:.1f}%)")
        return sigmoid(float(scale) * np.array(evs) + float(bias))

    # ── Run predictions on SV1 with cost tracking ─────────────────────────
    print("\n[10/10] Running predictions on SV1…")

    tracker = Tracker().start()

    # -- Test set evaluation on SV1
    print("   Running test set evaluation on SV1…")
    t0 = time.time()
    test_probs = predict_proba_sv1(X_test_q, sv1_weights, sv1_scale, sv1_bias)
    test_preds = (test_probs >= THRESHOLD).astype(int)
    print(f"   Test eval done in {time.time()-t0:.1f}s")

    acc = accuracy_score(y_test, test_preds)
    auc = roc_auc_score(y_test, test_probs)
    f1  = f1_score(y_test, test_preds, zero_division=0)
    cm  = confusion_matrix(y_test, test_preds)

    print(f"\n   ── SV1 EVALUATION METRICS (threshold={THRESHOLD:.2f}) ────────────")
    print(f"   Test Accuracy  : {acc*100:.1f}%")
    print(f"   ROC-AUC        : {auc:.4f}")
    print(f"   F1 Score       : {f1:.4f}")
    print(f"   Confusion Matrix: TN={cm[0,0]} FP={cm[0,1]} "
          f"FN={cm[1,0]} TP={cm[1,1]}")
    print(f"\n{classification_report(y_test, test_preds, target_names=['No Fire','Wildfire'], digits=3)}")

    # -- 2023 risk predictions on SV1
    print("   Running 2023 risk predictions on SV1…")
    t0 = time.time()
    pred_probs = predict_proba_sv1(X_pred_q, sv1_weights, sv1_scale, sv1_bias)
    print(f"   2023 predictions done in {time.time()-t0:.1f}s")

    tracker.stop()
    
    # Print actual AWS cost incurred
    try:
                # Print all tracked info
        print(tracker.tracked_resources())
        
        # Just the cost
        print(f"Cost so far: ${float(tracker.simulator_tasks_cost()):.6f}")
        
        # Quantity of tasks
        print(tracker.quantum_tasks_statistics())
    except Exception:
        print("\n   (Cost tracker unavailable — check AWS Cost Explorer)")

    # ── Format 2023 predictions ───────────────────────────────────────────
    df_2023 = df_2023.copy()
    df_2023["risk_probability"]   = pred_probs
    df_2023["risk_level"]         = pd.cut(
        pred_probs, bins=[0, 0.33, 0.55, 0.70, 1.0],
        labels=["Low", "Moderate", "High", "Extreme"])
    df_2023["wildfire_predicted"] = (pred_probs >= THRESHOLD).astype(int)

    at_risk = df_2023["wildfire_predicted"].sum()
    print(f"\n   Pred prob range: [{pred_probs.min():.3f}, {pred_probs.max():.3f}]")
    print(f"   ZIPs at fire risk 2023: {at_risk}/{len(df_2023)}")

    print("\n   ── 2023 RISK PREDICTIONS — TOP 40 (SV1) ───────────────────────")
    print(f"   {'ZIP':>8}  {'Prob':>8}  {'Level':>10}  {'Risk?':>6}")
    print("   " + "-" * 42)
    for _, r in df_2023.sort_values("risk_probability", ascending=False).head(40).iterrows():
        print(f"   {int(r['zip']):>8}  {r['risk_probability']:>8.3f}  "
              f"{str(r['risk_level']):>10}  {'Yes' if r['wildfire_predicted'] else 'No':>6}")

    # ── Save results ──────────────────────────────────────────────────────
    results = {
        "model":              "Hybrid Re-uploading VQC v7 (Local Train + SV1 Predict)",
        "train_backend":      "PennyLane default.qubit (local)",
        "predict_backend":    f"Amazon Braket SV1 — s3://{S3_BUCKET_NAME}",
        "n_qubits":           N_QUBITS,
        "n_layers":           N_LAYERS,
        "trainable_params":   TOTAL_PARAMS + 2,
        "best_scale":         round(sv1_scale, 4),
        "best_bias":          round(sv1_bias, 4),
        "decision_threshold": round(float(THRESHOLD), 2),
        "test_accuracy":      round(float(acc), 4),
        "roc_auc":            round(float(auc), 4),
        "f1_score":           round(float(f1), 4),
        "loss_histories":     [[round(float(l), 5) for l in h] for h in all_loss_histories],
        "predictions_2023": (
            df_2023[["zip", "risk_probability", "risk_level", "wildfire_predicted"]]
            .assign(risk_probability=lambda d: d["risk_probability"].round(4),
                    risk_level=lambda d: d["risk_level"].astype(str))
            .to_dict(orient="records")
        ),
    }

    out_path = "results/qml_results_v7.json"
    with open(out_path, "w") as f:
        json.dump(results, f, indent=2)

    print(f"\n   Results saved -> {out_path}")

print("\n" + "=" * 70)
print("  DONE")
print("=" * 70)


██████████████████████████████████████████████████████████████████████
  PHASE 2 — SV1 PREDICTIONS  (Amazon Braket, ~$1.50)
██████████████████████████████████████████████████████████████████████

[8/10] Loading weights from results/best_weights.npz…
   Weights loaded | scale=1.000 bias=-0.171

[9/10] Building SV1 device…

[10/10] Running predictions on SV1…
   Running test set evaluation on SV1…
   Running 2075 circuits...


In [23]:
# Test with just 5 samples first
tracker = Tracker().start()

test_small = predict_proba_sv1(X_test_q[:5], sv1_weights, sv1_scale, sv1_bias)

tracker.stop()

# Print all tracked info
print(tracker.tracked_resources())

# Just the cost
print(f"Cost so far: ${float(tracker.simulator_tasks_cost()):.6f}")

# Quantity of tasks
print(tracker.quantum_tasks_statistics())

   Running 5 circuits...
['arn:aws:braket:us-east-1:517306214684:quantum-task/70eea4c5-8ef1-4be8-b12c-0e0eb23dfc34', 'arn:aws:braket:us-east-1:517306214684:quantum-task/398f61c4-6db1-49c1-84e7-c27aba057e87', 'arn:aws:braket:us-east-1:517306214684:quantum-task/0d63c490-d1c4-48a8-b8f0-8a52cc073bf2', 'arn:aws:braket:us-east-1:517306214684:quantum-task/1e8544ca-fde6-4cf9-9a0b-f1acfad416e5', 'arn:aws:braket:us-east-1:517306214684:quantum-task/01c85d5f-b74f-4045-9a42-93ddd87db899']
Cost so far: $0.018750
{'arn:aws:braket:::device/quantum-simulator/amazon/sv1': {'shots': 0, 'tasks': {'COMPLETED': 5}, 'execution_duration': datetime.timedelta(microseconds=56000), 'billed_execution_duration': datetime.timedelta(seconds=15)}}


In [ ]:
import numpy as np
import pennylane.numpy as pnp
from sklearn.metrics import precision_recall_curve
from sklearn.utils.class_weight import compute_class_weight

ckpt = np.load("results/best_weights.npz")
weights = pnp.array(ckpt["weights"], requires_grad=False)

# FIX: extract 4 EVs per sample instead of collapsing to 1
def get_ev_features(X_q, weights):
    """Returns shape (n_samples, 4) — one EV per qubit."""
    evs = []
    for x in X_q:
        ev = quantum_circuit_local_multi(x, weights)
        evs.append([float(v) for v in ev])
    return np.array(evs)

# Sanity check: are the EVs actually different between classes?
X_check = get_ev_features(X_train_q, weights)
print(f"Feature shape: {X_check.shape}")
print(f"EV std across samples: {X_check.std(axis=0).round(4)}")
print(f"EV mean class=0: {X_check[y_train==0].mean(axis=0).round(4)}")
print(f"EV mean class=1: {X_check[y_train==1].mean(axis=0).round(4)}")
sep = abs(X_check[y_train==1].mean(axis=0) - X_check[y_train==0].mean(axis=0))
print(f"Class separation per qubit: {sep.round(4)}  (>0.05 = useful signal)")


In [ ]:
X_train_ev = get_ev_features(X_train_q, weights)
X_val_ev   = get_ev_features(X_val_q, weights)
X_test_ev  = get_ev_features(X_test_q, weights)
print(f"Train EV: {X_train_ev.shape}, Val EV: {X_val_ev.shape}, Test EV: {X_test_ev.shape}")


In [ ]:
from sklearn.svm import SVC

# FIX: class_weight='balanced' corrects for the 10.9x imbalance
cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
cw_dict = dict(zip(np.unique(y_train), cw))

svm_model = SVC(probability=True, kernel='rbf', class_weight=cw_dict, C=10, gamma='scale')
svm_model.fit(X_train_ev, y_train)

val_probs = svm_model.predict_proba(X_val_ev)[:, 1]

# FIX: use PR curve for principled threshold selection
prec, rec, thresholds = precision_recall_curve(y_val, val_probs)
f1s = 2 * prec * rec / (prec + rec + 1e-9)
best_t = float(thresholds[np.argmax(f1s[:-1])]) if len(thresholds) else 0.5

test_probs = svm_model.predict_proba(X_test_ev)[:, 1]
test_preds = (test_probs >= best_t).astype(int)

print(f"=== SVM (balanced, 4-qubit features, threshold={best_t:.2f}) ===")
print("Accuracy:", round(accuracy_score(y_test, test_preds), 4))
print("ROC-AUC :", round(roc_auc_score(y_test, test_probs), 4))
print("F1      :", round(f1_score(y_test, test_preds), 4))
print(classification_report(y_test, test_preds, target_names=['No Fire','Wildfire'], digits=3))


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
rf_model.fit(X_train_ev, y_train)

val_probs_rf = rf_model.predict_proba(X_val_ev)[:, 1]

# FIX: use PR curve on rf_model probabilities (not svm_model)
prec, rec, thresholds = precision_recall_curve(y_val, val_probs_rf)
f1s = 2 * prec * rec / (prec + rec + 1e-9)
best_t_rf = float(thresholds[np.argmax(f1s[:-1])]) if len(thresholds) else 0.5

# FIX: evaluate rf_model on test set (was incorrectly using svm_model)
test_probs_rf = rf_model.predict_proba(X_test_ev)[:, 1]
test_preds_rf = (test_probs_rf >= best_t_rf).astype(int)

print(f"=== Random Forest (balanced, 4-qubit features, threshold={best_t_rf:.2f}) ===")
print("Accuracy:", round(accuracy_score(y_test, test_preds_rf), 4))
print("ROC-AUC :", round(roc_auc_score(y_test, test_probs_rf), 4))
print("F1      :", round(f1_score(y_test, test_preds_rf), 4))
print(classification_report(y_test, test_preds_rf, target_names=['No Fire','Wildfire'], digits=3))


In [ ]:
from xgboost import XGBClassifier

scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,   # FIX: handles imbalance in XGB
    eval_metric='logloss',
    random_state=42
)
xgb_model.fit(X_train_ev, y_train)

val_probs_xgb = xgb_model.predict_proba(X_val_ev)[:, 1]

# FIX: use PR curve on xgb_model probabilities (not svm_model)
prec, rec, thresholds = precision_recall_curve(y_val, val_probs_xgb)
f1s = 2 * prec * rec / (prec + rec + 1e-9)
best_t_xgb = float(thresholds[np.argmax(f1s[:-1])]) if len(thresholds) else 0.5

# FIX: evaluate xgb_model on test set (was incorrectly using svm_model)
test_probs_xgb = xgb_model.predict_proba(X_test_ev)[:, 1]
test_preds_xgb = (test_probs_xgb >= best_t_xgb).astype(int)

print(f"=== XGBoost (scale_pos_weight={scale_pos_weight:.1f}, 4-qubit features, threshold={best_t_xgb:.2f}) ===")
print("Accuracy:", round(accuracy_score(y_test, test_preds_xgb), 4))
print("ROC-AUC :", round(roc_auc_score(y_test, test_probs_xgb), 4))
print("F1      :", round(f1_score(y_test, test_preds_xgb), 4))
print(classification_report(y_test, test_preds_xgb, target_names=['No Fire','Wildfire'], digits=3))

# ── Summary comparison ─────────────────────────────────────────────────────
print("\n=== MODEL COMPARISON SUMMARY ===")
print(f"{'Model':<25} {'AUC':>8} {'F1':>8}")
print("-" * 44)
print(f"{'SVM (balanced)':>25}  {roc_auc_score(y_test, test_probs):>8.4f} {f1_score(y_test, test_preds):>8.4f}")
print(f"{'RF (balanced)':>25}   {roc_auc_score(y_test, test_probs_rf):>8.4f} {f1_score(y_test, test_preds_rf):>8.4f}")
print(f"{'XGB (scale_pos_weight)':>25} {roc_auc_score(y_test, test_probs_xgb):>8.4f} {f1_score(y_test, test_preds_xgb):>8.4f}")


## Summary of fixes in this version

### Bugs fixed
| Cell | Bug | Fix |
|------|-----|-----|
| 14   | `svm_model.predict_proba()` used for RF evaluation | Changed to `rf_model.predict_proba()` |
| 15   | `svm_model.predict_proba()` used for XGB evaluation | Changed to `xgb_model.predict_proba()` |

### Model improvements
| Cell | Change | Reason |
|------|--------|--------|
| 1    | `INIT_SCALE` 0.05 → `np.pi` | The near-zero init caused a barren plateau; weights never moved |
| 1    | `N_EPOCHS` 100 → 150 | More training time now that gradients are non-zero |
| 1    | `L2_BIAS` 0.01 → 0.001 | Was over-constraining the bias parameter |
| 8    | Added `quantum_circuit_local_multi` | Returns EV from all 4 qubits instead of qubit-0 only |
| 11   | `get_ev_features` now returns shape `(n_samples, 4)` | 4× more quantum information fed to classifiers |
| 13   | `SVC(class_weight=balanced)` | Was ignoring the 10.9× class imbalance |
| 14   | `RF(class_weight=balanced)` | Same fix |
| 15   | `XGB(scale_pos_weight=10.9)` | Same fix for XGBoost |
| 13–15 | PR-curve threshold selection | More principled than grid sweep over `[0.20, 0.81]` |
